# Qwen3.5 Audit v11 CVS Context v3.1

This notebook derives `v3.1` from the cached `cvsctx_v3__debug5` results.

Rule:
- For any method row whose clip-level right-tool prediction is exactly `Hook`, consult the saved best rubric prediction from `qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps.ipynb`.
- If the rubric prediction is `Maryland`, rewrite right-hand `Hook` tool labels in that row to `Maryland`.
- Otherwise leave the `v3` prediction unchanged.

This is applied to every `cvsctx_v3` method (`description`, `hint`, `structured`, `structured_deterministic`, etc.) present in the source results.


In [1]:
import json
import sys
from copy import deepcopy
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cvs_act.action_segment_eval import (
    convert_record_to_simple_actions,
    evaluate_method_predictions,
    load_audit_records,
    naturalize_simple_actions,
    read_json,
    write_json,
)

ANNOTATION_ROOT = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1'
AUDIT_V11_DIR = ANNOTATION_ROOT / 'audit_v11'
SOURCE_ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3__debug5'
RUBRIC_CACHE_PATH = ROOT_DIR / 'notebooks/qwen3.5_audit_v11_seg_extract_spec_eval/artifacts/qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps_cache.json'
ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_1'
RESULTS_PATH = ARTIFACT_DIR / 'results.json'
SPEC_EVAL_PATH = ARTIFACT_DIR / 'spec_eval_results.json'
SUMMARY_PATH = ARTIFACT_DIR / 'override_summary.json'
SYNTHETIC_GT_DEFAULT_METHOD = 'structured_prediction_cvs_context'
RUBRIC_METHOD = 'prompt2_all_in_one_json_with_shapes'
LABELING_VERSION = 'code_labels_v3_1_cvsctx_from_v3_debug5_hook_to_maryland'
WRITE_DERIVED_ARTIFACTS = True

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('source:', SOURCE_ARTIFACT_DIR)
print('rubric cache:', RUBRIC_CACHE_PATH)
print('artifacts:', ARTIFACT_DIR)


source: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3__debug5
rubric cache: /mnt/md0/weiqiuy/surgent/notebooks/qwen3.5_audit_v11_seg_extract_spec_eval/artifacts/qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps_cache.json
artifacts: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_1__debug5


In [2]:
records = load_audit_records(AUDIT_V11_DIR)
gt_simple_records = [convert_record_to_simple_actions(record) for record in records]
source_results = read_json(SOURCE_ARTIFACT_DIR / 'results.json', {'results': []})['results']
rubric_cache = read_json(RUBRIC_CACHE_PATH, {})


def normalize_tool_label(value):
    if value is None:
        return ''
    value = str(value).strip()
    if not value:
        return ''
    mapping = {
        'grasper': 'Grasper',
        'maryland': 'Maryland',
        'hook': 'Hook',
        'irrigator': 'Irrigator',
        'scissors': 'Scissors',
        'clipper': 'Clipper',
        'unknown': 'Unknown',
        '(absent)': '(absent)',
        '(not set)': '(absent)',
        'not set': '(absent)',
        'none': '(absent)',
    }
    return mapping.get(value.lower(), value)


def clip_tool_from_segments(segments):
    tools = sorted({
        normalize_tool_label(seg.get('tool_type'))
        for seg in (segments or [])
        if normalize_tool_label(seg.get('tool_type'))
    })
    if not tools:
        return '(absent)', '(absent)'
    if len(tools) == 1:
        return tools[0], tools[0]
    return '(multi)', ' | '.join(tools)


def get_rubric_tool(example_id):
    payload = rubric_cache.get(example_id, {}).get(RUBRIC_METHOD, {})
    return normalize_tool_label((payload.get('aggregation') or {}).get('predicted_tool', ''))


def rewrite_right_hook_to_maryland(extracted_actions):
    updated = deepcopy(extracted_actions)
    changed = False
    for seg in updated.get('right', []) or []:
        if normalize_tool_label(seg.get('tool_type')) != 'Hook':
            continue
        seg['tool_type'] = 'Maryland'
        triplet = list(seg.get('triplet') or [])
        if triplet:
            triplet[0] = 'Maryland'
            seg['triplet'] = triplet
        changed = True
    return updated, changed


methods = sorted({row.get('method') for row in source_results if row.get('status') == 'ok'})
print('source rows:', len(source_results))
print('methods:', methods)


source rows: 360
methods: ['description_only_cvs_context', 'hint_questions_cvs_context', 'structured_prediction_cvs_context', 'structured_prediction_deterministic_cvs_context']


In [3]:
derived_results = []
override_rows = []

for row in source_results:
    item = deepcopy(row)
    if item.get('status') != 'ok':
        derived_results.append(item)
        continue

    clip_tool_name, clip_tool_set_text = clip_tool_from_segments(item.get('extracted_actions', {}).get('right', []))
    rubric_tool_name = get_rubric_tool(item['example_id'])
    applied = False

    if clip_tool_name == 'Hook' and rubric_tool_name == 'Maryland':
        updated_actions, changed = rewrite_right_hook_to_maryland(item['extracted_actions'])
        if changed:
            item['extracted_actions'] = updated_actions
            item['extracted_actions_natural_language'] = naturalize_simple_actions(updated_actions)
            applied = True

    item['labeling_version'] = LABELING_VERSION
    item['v3_1_override'] = {
        'source_result_dir': str(SOURCE_ARTIFACT_DIR),
        'rule': 'if clip-level v3 right tool is Hook and best rubric clip tool is Maryland, rewrite right Hook tool labels to Maryland',
        'source_clip_right_tool': clip_tool_name,
        'source_clip_right_tool_set_text': clip_tool_set_text,
        'rubric_method': RUBRIC_METHOD,
        'rubric_clip_tool': rubric_tool_name,
        'applied': applied,
    }
    derived_results.append(item)

    override_rows.append({
        'example_id': item.get('example_id'),
        'video_id': item.get('video_id'),
        'criterion': item.get('criterion'),
        'method': item.get('method'),
        'source_clip_right_tool': clip_tool_name,
        'rubric_clip_tool': rubric_tool_name,
        'override_applied': applied,
    })

if WRITE_DERIVED_ARTIFACTS:
    write_json(RESULTS_PATH, {'results': derived_results})
    write_json(SUMMARY_PATH, override_rows)

    synthetic_gt_by_method = {}
    for item in derived_results:
        if item.get('status') != 'ok':
            continue
        method_name = item.get('method')
        if not method_name:
            continue
        pred = dict(item.get('extracted_actions', {}))
        pred['example_id'] = item['example_id']
        pred['video_id'] = item['video_id']
        pred['criterion'] = item.get('criterion')
        synthetic_gt_by_method.setdefault(method_name, []).append(pred)

    for method_name, records in synthetic_gt_by_method.items():
        method_slug = method_name.replace('/', '__')
        write_json(ARTIFACT_DIR / f'synthetic_audit_v11_simple_actions__{method_slug}.json', records)

    default_records = synthetic_gt_by_method.get(SYNTHETIC_GT_DEFAULT_METHOD)
    if default_records:
        write_json(ARTIFACT_DIR / 'synthetic_audit_v11_simple_actions.json', default_records)

summary_df = pd.DataFrame(override_rows)
display(summary_df.groupby(['method', 'source_clip_right_tool', 'rubric_clip_tool'], dropna=False)['override_applied'].agg(['count', 'sum']).reset_index().sort_values(['method', 'sum', 'count'], ascending=[True, False, False]))
print('saved results:', RESULTS_PATH)


,method,source_clip_right_tool,rubric_clip_tool,count,sum
12,description_only_cvs_context,Hook,Maryland,16,16
10,description_only_cvs_context,Hook,Hook,33,0
11,description_only_cvs_context,Hook,Irrigator,8,0
13,description_only_cvs_context,Hook,Scissors,8,0
9,description_only_cvs_context,Hook,Clipper,6,0
2,description_only_cvs_context,(absent),Maryland,3,0
7,description_only_cvs_context,(multi),Maryland,3,0
15,description_only_cvs_context,Maryland,Maryland,3,0
3,description_only_cvs_context,(absent),Scissors,2,0
5,description_only_cvs_context,(multi),Hook,2,0


saved results: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_1__debug5/results.json


In [4]:
ok_results = [item for item in derived_results if item.get('status') == 'ok']
spec_eval = {}
for method_name in methods:
    pred_records = []
    for item in ok_results:
        if item['method'] != method_name:
            continue
        pred = dict(item['extracted_actions'])
        pred['example_id'] = item['example_id']
        pred['video_id'] = item['video_id']
        pred['criterion'] = item.get('criterion')
        pred_records.append(pred)
    spec_eval[method_name] = evaluate_method_predictions(pred_records, gt_simple_records)

if WRITE_DERIVED_ARTIFACTS:
    write_json(SPEC_EVAL_PATH, spec_eval)

rows = []
for method_name, method_rows in spec_eval.items():
    for row in method_rows:
        rows.append({k: v for k, v in {'method': method_name, **row}.items() if k != 'details'})

spec_df = pd.DataFrame(rows)
overall = (
    spec_df.groupby('method', as_index=False)
    .agg(mean_f1=('f1_mean', 'mean'), rows=('f1_mean', 'size'))
    .sort_values('mean_f1', ascending=False)
)
actor = (
    spec_df.groupby(['actor', 'method'], as_index=False)
    .agg(mean_f1=('f1_mean', 'mean'))
    .pivot(index='actor', columns='method', values='mean_f1')
    .reset_index()
)

display(overall)
display(actor)


,method,mean_f1,rows
2,structured_prediction_cvs_context,0.329713,30
1,hint_questions_cvs_context,0.282909,30
3,structured_prediction_deterministic_cvs_context,0.282570,30
0,description_only_cvs_context,0.207253,30


method,actor,description_only_cvs_context,hint_questions_cvs_context,structured_prediction_cvs_context,structured_prediction_deterministic_cvs_context
0,camera,0.016314,0.130742,0.167994,0.153000
1,left,0.143423,0.413602,0.587771,0.555643
2,other,1.000000,0.555556,0.333333,0.000000
3,right,0.197775,0.213500,0.232169,0.233257
